# E1/E2 谄媚切片 Kaggle 跑数（qwen3:8b，零 API 额度）

前置：三个 Dataset 已挂载（见 kaggle/PORT.md）——
- `/kaggle/input/dsh-mneme-src`（dsh-mneme 包源码，无 node_modules）
- `/kaggle/input/persistbench-harness`（脚本 + protocol + data）
- `/kaggle/input/persistbench-cached`（旧 results jsonl，A 臂缓存）

流程：SMOKE → E1_PILOT → E1_FULL；E2_LABEL → E2_RUN。断点续跑见 PORT.md。
GPU 会话（T4），12h 上限。

In [ ]:
%%bash
# ---- 1) Node 24 via tarball（不依赖 conda；node:sqlite 免 flag）----
set -x
NODE_VER=24.15.0
curl -fsSL "https://nodejs.org/dist/v${NODE_VER}/node-v${NODE_VER}-linux-x64.tar.xz" -o /tmp/node.tar.xz &&   tar -xJf /tmp/node.tar.xz -C /usr/local --strip-components=1 || echo NODE_TARBALL_FAILED
node -v
node -e "new (require('node:sqlite').DatabaseSync)(':memory:'); console.log('node:sqlite ok')" || echo NODE_SQLITE_FAILED
nvidia-smi | head -12 || echo NO_GPU

In [ ]:
%%bash
# ---- 2) Ollama:先补 zstd,再 install.sh,失败留全量日志 ----
set -x
mkdir -p /kaggle/working/logs
(apt-get update > /kaggle/working/logs/apt.log 2>&1; apt-get install -y zstd >> /kaggle/working/logs/apt.log 2>&1) || echo APT_FAILED
which zstd || echo ZSTD_MISSING
curl -fsSL https://ollama.com/install.sh -o /tmp/oi.sh && bash /tmp/oi.sh > /kaggle/working/logs/oi2.log 2>&1 || echo INSTALL_SH_FAILED
which ollama && ollama --version || { echo OLLAMA_STILL_MISSING; tail -8 /kaggle/working/logs/oi2.log /kaggle/working/logs/apt.log 2>/dev/null; }
exit 0


In [ ]:
# ---- 2b) serve + pull（ollama 缺席时不炸，诊断落地）----
import subprocess, time
diag = {}
if subprocess.call(["which", "ollama"]) == 0:
    subprocess.Popen(["ollama", "serve"], stdout=open("/kaggle/working/ollama.log", "w"), stderr=subprocess.STDOUT)
    print("ollama serve started")
    for _ in range(30):
        if subprocess.call(["curl", "-s", "http://127.0.0.1:11434/api/version"], stdout=subprocess.DEVNULL) == 0:
            print("API_UP"); break
        time.sleep(2)
    r = subprocess.run(["ollama", "pull", "qwen3:8b"], capture_output=True, text=True, timeout=1200)
    diag["pull_tail"] = (r.stdout + r.stderr).strip().splitlines()[-3:]
    print("pull:", diag["pull_tail"])
else:
    diag["pull_tail"] = ["OLLAMA MISSING — 后续跑数必然失败"]
    print(diag["pull_tail"][0])
open("/kaggle/working/logs/pull-diag.txt", "w").write(repr(diag))
!curl -s http://127.0.0.1:11434/api/tags | head -c 300

In [ ]:
# ---- 3) 工作区（/kaggle/tmp，不持久化——/kaggle/working 只放结果）+ 运行时 ----
import shutil, os, traceback
!find /kaggle/input -maxdepth 4 | head -40
WS = "/kaggle/tmp/mneme-stack"
open("/kaggle/working/WS_PATH.txt", "w").write(WS)
os.makedirs(WS, exist_ok=True)
def guard(name, fn):
    try:
        fn(); print(f"[ok] {name}")
    except Exception as e:
        print(f"[FAIL] {name}: {e!r}")
        traceback.print_exc()
def find_dir(marker):
    for root, dirs, files in os.walk("/kaggle/input"):
        if marker in files:
            return root
    return None
src_dir = find_dir("package.json")
harness_dir = find_dir("run-sycophancy.mjs")
print("discovered:", src_dir, "|", harness_dir)
def do_copy():
    assert src_dir and harness_dir, f"dataset mount not found: src={src_dir} harness={harness_dir}"
    shutil.copytree(src_dir, f"{WS}/dsh-mneme/dsh-mneme", dirs_exist_ok=True)
    shutil.copytree(harness_dir, f"{WS}/persistbench-mneme", dirs_exist_ok=True)
    cached_gen = find_dir("results-sycophancy-gen.jsonl")
    if cached_gen:
        for f in ("results-sycophancy-gen.jsonl", "results-sycophancy-judge.jsonl"):
            p_ = os.path.join(cached_gen, f)
            if os.path.exists(p_): shutil.copy(p_, f"{WS}/persistbench-mneme/")
    labels_dir = find_dir("sycophancy-epistemic-labels.jsonl")
    print("labels_dir:", labels_dir)
    if labels_dir:
        shutil.copy(os.path.join(labels_dir, "sycophancy-epistemic-labels.jsonl"), f"{WS}/persistbench-mneme/data/")
    print("staged:", sorted(os.listdir(f"{WS}/persistbench-mneme")))
guard("copytree", do_copy)
!cd {WS}/dsh-mneme/dsh-mneme && npm install --no-audit --no-fund > /tmp/npm.log 2>&1; tail -1 /tmp/npm.log; echo NPM_DONE
!mkdir -p /root/hf-runtime && cd /root/hf-runtime && npm init -y > /dev/null 2>&1; npm i @huggingface/transformers@^4.2.0 --no-audit --no-fund > /tmp/hf.log 2>&1; tail -1 /tmp/hf.log; echo HF_DONE
!cd {WS}/dsh-mneme/dsh-mneme && node scripts/mneme-runtime.mjs adopt --from /root/hf-runtime/node_modules 2>&1 | tail -3 || echo ADOPT_FAILED
!cd {WS}/dsh-mneme/dsh-mneme && node scripts/mneme-runtime.mjs verify 2>&1 | tail -3 || echo VERIFY_FAILED
!cd {WS}/persistbench-mneme && node -e "const {createRequire}=require('node:module');const r=createRequire(process.cwd()+'/');const {LocalEmbedder}=r('../dsh-mneme/dsh-mneme/src/local-embedder.js');(async()=>{const e=new LocalEmbedder({});await e.init?.().catch?.(()=>{});const v=await e.embedSingle('warmup');console.log('embedder ready dim='+v.length)})()" 2>&1 | tail -2 || echo EMBEDDER_FAILED

In [ ]:
# ---- 4) v12:E2 四臂重跑(修复后的包,R+ 重排应真实生效)(标注来自 persistbench-cached dataset v2)----
import subprocess, os
MNEME = "/kaggle/tmp/mneme-stack/dsh-mneme/dsh-mneme"
LIVE = "/kaggle/working/results/live"
os.makedirs(LIVE, exist_ok=True)
cmd = (f"cd /kaggle/tmp/mneme-stack/persistbench-mneme && "
       f"node run-sycophancy.mjs --mneme {MNEME} --mode epistemic --limit 200 "
       f"--labels data/sycophancy-epistemic-labels.jsonl "
       f"--gen {LIVE}/results-e2b-gen.jsonl --judge {LIVE}/results-e2b-judge.jsonl; echo E2B_RUN_EXIT=$?")
print("===== E2B start =====", flush=True)
r = subprocess.run(["bash", "-lc", cmd], capture_output=True, text=True)
tail = (r.stdout + r.stderr).strip().splitlines()
for l in tail[-14:]:
    print(l[:200], flush=True)
print(f"===== E2B done rc={r.returncode} =====", flush=True)
open("/kaggle/working/E2B_RUN_exit.txt", "w").write(str(r.returncode))

## 会话被掐后（12h 上限）

1. 下载 `/kaggle/working/mneme-stack/persistbench-mneme/results-*.jsonl`；
2. 上传成 `persistbench-cached` Dataset 新版本（或并入新 Dataset）；
3. 新会话重跑同一条命令——行键 `idx:arm` 已完成的自动跳过；
4. judge 的 null 行先 `grep -v '"score":null' results-e1-judge.jsonl` 清理再用更大
   `--judgetok` 补跑（本机同款坑）。

结果取回本地后：分析走 `analysis-sycophancy-threshold.ipynb`（配对差口径），
读数写 RESULTS.md 分段 + #280 回复草稿追加。

In [ ]:
# ---- 5) 收尾：results + 诊断拷进 /kaggle/working ----
import shutil, glob, os, subprocess
out = "/kaggle/working/results"
os.makedirs(out, exist_ok=True)
n = 0
for f in glob.glob("/kaggle/tmp/mneme-stack/persistbench-mneme/results-*.jsonl"):
    shutil.copy(f, out); n += 1
def sh(c):
    return subprocess.run(["bash", "-lc", c], capture_output=True, text=True).stdout.strip() or "EMPTY/FAIL"
diag = {
    "node": sh("node -v"),
    "ollama": sh("which ollama && ollama --version"),
    "ollama_api": sh("curl -s --max-time 3 http://127.0.0.1:11434/api/tags | head -c 200"),
    "runtime_payloads": sh("ls /root/.dsh/mneme/runtime/ 2>/dev/null || ls ~/.dsh/mneme/runtime/ 2>/dev/null"),
    "results_copied": n,
}
open("/kaggle/working/diagnostics.txt", "w").write(repr(diag))
print(diag)
print("results:", os.listdir(out))